<a href="https://colab.research.google.com/github/iamhardyyy/git-demo2/blob/main/Next%20Word%20prediction%20using%20bidirectional%20LSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import os

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Embedding,LSTM,Dense,Bidirectional
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam

In [ ]:
data=pd.read_csv('/content/drive/MyDrive/medium_data.csv')
data

,id,url,title,subtitle,image,claps,responses,reading_time,publication,date
0,1,https://towardsdatascience.com/a-beginners-gui...,A Beginner’s Guide to Word Embedding with Gens...,NaN,1.png,850,8,8,Towards Data Science,2019-05-30
1,2,https://towardsdatascience.com/hands-on-graph-...,Hands-on Graph Neural Networks with PyTorch & ...,NaN,2.png,1100,11,9,Towards Data Science,2019-05-30
2,3,https://towardsdatascience.com/how-to-use-ggpl...,How to Use ggplot2 in Python,A Grammar of Graphics for Python,3.png,767,1,5,Towards Data Science,2019-05-30
3,4,https://towardsdatascience.com/databricks-how-...,Databricks: How to Save Files in CSV on Your L...,When I work on Python projects dealing…,4.jpeg,354,0,4,Towards Data Science,2019-05-30
4,5,https://towardsdatascience.com/a-step-by-step-...,A Step-by-Step Implementation of Gradient Desc...,One example of building neural…,5.jpeg,211,3,4,Towards Data Science,2019-05-30
...,...,...,...,...,...,...,...,...,...,...
6503,6504,https://medium.com/better-marketing/we-vs-i-ho...,“We” vs “I” — How Should You Talk About Yourse...,Basic copywriting choices with a big…,6504.jpg,661,6,6,Better Marketing,2019-12-05
6504,6505,https://medium.com/better-marketing/how-donald...,How Donald Trump Markets Himself,Lessons from who might be the most popular bra...,6505.jpeg,189,1,5,Better Marketing,2019-12-05
6505,6506,https://medium.com/better-marketing/content-an...,Content and Marketing Beyond Mass Consumption,How to acquire customers without wasting money...,6506.jpg,207,1,8,Better Marketing,2019-12-05
6506,6507,https://medium.com/better-marketing/5-question...,5 Questions All Copywriters Should Ask Clients...,Save time and effort by…,6507.jpg,253,2,5,Better Marketing,2019-12-05


In [ ]:
data['title']

,title
0,A Beginner’s Guide to Word Embedding with Gens...
1,Hands-on Graph Neural Networks with PyTorch & ...
2,How to Use ggplot2 in Python
3,Databricks: How to Save Files in CSV on Your L...
4,A Step-by-Step Implementation of Gradient Desc...
...,...
6503,“We” vs “I” — How Should You Talk About Yourse...
6504,How Donald Trump Markets Himself
6505,Content and Marketing Beyond Mass Consumption
6506,5 Questions All Copywriters Should Ask Clients...


In [ ]:
data['title']=data['title'].apply(lambda x:x.replace(u'\xa0',u''))
data['title']=data['title'].apply(lambda x:x.replace('\u200a',''))

In [ ]:
tokenize=Tokenizer()
tokenize.fit_on_texts(data['title'])
total_words=len(tokenize.word_index)+1
total_words

10863

In [ ]:
input_num= []
for line in data['title']:
    token = tokenize.texts_to_sequences([line])[0]
    #print(token_list)

    for i in range(1, len(token)):
        n_gram = token[:i+1]
        input_num.append(n_gram)

print(input_num)
print("Total input sequences: ", len(input_num))

[[4, 565], [4, 565, 60], [4, 565, 60, 1], [4, 565, 60, 1, 434], [4, 565, 60, 1, 434, 1311], [4, 565, 60, 1, 434, 1311, 14], [4, 565, 60, 1, 434, 1311, 14, 3517], [4, 565, 60, 1, 434, 1311, 14, 3517, 3518], [3519, 21], [3519, 21, 783], [3519, 21, 783, 111], [3519, 21, 783, 111, 146], [3519, 21, 783, 111, 146, 14], [3519, 21, 783, 111, 146, 14, 476], [3519, 21, 783, 111, 146, 14, 476, 476], [3519, 21, 783, 111, 146, 14, 476, 476, 1654], [5, 1], [5, 1, 62], [5, 1, 62, 3520], [5, 1, 62, 3520, 193], [3521, 5], [3521, 5, 1], [3521, 5, 1, 232], [3521, 5, 1, 232, 1079], [3521, 5, 1, 232, 1079, 10], [3521, 5, 1, 232, 1079, 10, 2220], [3521, 5, 1, 232, 1079, 10, 2220, 21], [3521, 5, 1, 232, 1079, 10, 2220, 21, 9], [3521, 5, 1, 232, 1079, 10, 2220, 21, 9, 3522], [4, 170], [4, 170, 63], [4, 170, 63, 170], [4, 170, 63, 170, 400], [4, 170, 63, 170, 400, 6], [4, 170, 63, 170, 400, 6, 3523], [4, 170, 63, 170, 400, 6, 3523, 2221], [4, 170, 63, 170, 400, 6, 3523, 2221, 7], [4, 170, 63, 170, 400, 6, 3523

In [ ]:
max_len=max([len(x) for x in input_num])

input_num = np.array(pad_sequences(input_num, maxlen=max_len, padding='pre'))
input_num[1]

array([  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   4, 565,  60],
      dtype=int32)

In [ ]:
xs=input_num[:,:-1]
label=input_num[:,-1]
ys=tf.keras.utils.to_categorical(label,num_classes=total_words)

In [ ]:
model = Sequential()
model.add(Embedding(total_words, 100, input_length=max_len-1))
model.add(Bidirectional(LSTM(150)))
model.add(Dense(total_words, activation='softmax'))
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
word = model.fit(xs, ys, epochs=60, verbose=1)
print(model)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Epoch 1/60
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 25s 13ms/step - accuracy: 0.0818 - loss: 7.4619
Epoch 2/60
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 18s 13ms/step - accuracy: 0.1299 - loss: 6.6191
Epoch 3/60
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 18s 13ms/step - accuracy: 0.1531 - loss: 6.0571
Epoch 4/60
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 19s 14ms/step - accuracy: 0.1749 - loss: 5.5111
Epoch 5/60
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 18s 13ms/step - accuracy: 0.1949 - loss: 4.9753
Epoch 6/60
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 20s 14ms/step - accuracy: 0.2260 - loss: 4.4688
Epoch 7/60
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 19s 14ms/step - accuracy: 0.2764 - loss: 4.0021
Epoch 8/60
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 18s 13ms/step - accuracy: 0.3421 - loss: 3.5667
Epoch 9/60
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 18s 13ms/step - accuracy: 0.4062 - loss: 3.1732
Epoch 10/60
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 20s 13ms/step - accuracy: 0.4646 - loss: 2.8267
Epoch 11/60
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 18s 13ms/step - accuracy: 0.5153 - loss: 2.5242
Epoch 12

In [ ]:
text = input('enter the word :')
next_words =int(input('number of words to be predicted :'))

for _ in range(next_words):
    token_list = tokenize.texts_to_sequences([text])[0]
    token_list = pad_sequences([token_list], maxlen=max_len-1, padding='pre')
    predictions = model.predict(token_list, verbose=0)
    predicted_index = np.argmax(predictions[0])
    output_word = tokenize.index_word.get(predicted_index, '')
    text += " " + output_word

print(text)

enter the word :deep
number of words to be predicted :5
deep learning for clinical diagnostics classification


In [ ]:
import pickle, json

os.makedirs('model', exist_ok=True)

model.save('model/lstm_model.h5')
pickle.dump(tokenize, open('model/tokenizer.pkl', 'wb'))
json.dump({'max_len': max_len}, open('model/config.json', 'w'))

In [ ]:
import keras
print(keras.__version__)

3.13.2
